# Chapter 6: Fine-tuning for Classification

Companion notebook for [Chapter 6 of the interactive course](https://bc-sparknet.github.io/make-an-llm-2/#/chapter/classification). The code cells follow the chapter's examples in order; the explanations are on the site.

*This notebook is generated from the chapter source. To change it, edit the chapter and run `npm run notebooks`.*

In [ ]:
%pip install -q torch tiktoken

## Setup

Code carried over from earlier chapters, plus anything this chapter needs to run. Run it once and move on.

In [ ]:
#@title Setup (run me first)

import math
import tiktoken
import torch
import torch.nn as nn
import torch.nn.functional as F

# ---- from Chapter 2 ----
import tiktoken
import torch
from torch.utils.data import Dataset, DataLoader

class NextTokenDataset(Dataset):
    def __init__(self, text, tokenizer, context_length, stride):
        ids = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
        self.inputs, self.targets = [], []
        for start in range(0, len(ids) - context_length, stride):
            window = ids[start : start + context_length + 1]
            self.inputs.append(torch.tensor(window[:-1]))   # all but the last
            self.targets.append(torch.tensor(window[1:]))   # all but the first

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, i):
        return self.inputs[i], self.targets[i]


def create_dataloader(text, batch_size=8, max_length=256, stride=256,
                      shuffle=True, drop_last=True):
    dataset = NextTokenDataset(text, tiktoken.get_encoding("gpt2"),
                               context_length=max_length, stride=stride)
    return DataLoader(dataset, batch_size=batch_size,
                      shuffle=shuffle, drop_last=drop_last)

# ---- from Chapter 3 ----
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads
        self.to_q = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.to_k = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.to_v = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        mask = torch.triu(torch.ones(context_length, context_length, dtype=torch.bool), diagonal=1)
        self.register_buffer("mask", mask)

    def split_heads(self, t):
        b, n, _ = t.shape
        # (b, n, d_out) -> (b, n, heads, head_dim) -> (b, heads, n, head_dim)
        return t.view(b, n, self.num_heads, self.head_dim).transpose(1, 2)

    def forward(self, x):
        b, n, _ = x.shape
        q = self.split_heads(self.to_q(x))
        k = self.split_heads(self.to_k(x))
        v = self.split_heads(self.to_v(x))

        scores = q @ k.transpose(2, 3)                    # (b, heads, n, n)
        scores = scores.masked_fill(self.mask[:n, :n], float('-inf'))
        weights = torch.softmax(scores / self.head_dim ** 0.5, dim=-1)
        weights = self.dropout(weights)

        out = weights @ v                                 # (b, heads, n, head_dim)
        out = out.transpose(1, 2).contiguous().view(b, n, -1)  # concat heads
        return self.out_proj(out)                         # (b, n, d_out)

# ---- from Chapter 4 ----
GPT_CONFIG_124M = {
    "vocab_size": 50257,     # number of tokens the BPE tokenizer knows
    "context_length": 1024,  # longest input the model can look at
    "emb_dim": 768,          # length of every token vector
    "n_heads": 12,           # attention heads per block
    "n_layers": 12,          # how many transformer blocks to stack
    "drop_rate": 0.1,        # dropout probability
    "qkv_bias": True,        # bias terms in the query/key/value projections
}

class LayerNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.scale = nn.Parameter(torch.ones(dim))   # γ
        self.shift = nn.Parameter(torch.zeros(dim))  # β

    def forward(self, x):
        mu = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        x_hat = (x - mu) / torch.sqrt(var + self.eps)
        return self.scale * x_hat + self.shift

import math

class GELU(nn.Module):
    def forward(self, x):
        c = math.sqrt(2.0 / math.pi)
        return 0.5 * x * (1.0 + torch.tanh(c * (x + 0.044715 * x ** 3)))

class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        d = cfg["emb_dim"]
        self.net = nn.Sequential(
            nn.Linear(d, 4 * d),  # expand:  768 → 3072
            GELU(),
            nn.Linear(4 * d, d),  # project: 3072 → 768
        )

    def forward(self, x):
        return self.net(x)

class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        d = cfg["emb_dim"]
        self.norm1 = LayerNorm(d)
        self.attn = MultiHeadAttention(
            d_in=d,
            d_out=d,
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"],
        )
        self.norm2 = LayerNorm(d)
        self.ff = FeedForward(cfg)
        self.drop = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        # First half: tokens exchange information (attention) + shortcut
        x = x + self.drop(self.attn(self.norm1(x)))
        # Second half: each token is processed on its own (FFN) + shortcut
        x = x + self.drop(self.ff(self.norm2(x)))
        return x

class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop = nn.Dropout(cfg["drop_rate"])
        self.blocks = nn.Sequential(*[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, token_ids):
        n_tokens = token_ids.shape[1]
        positions = torch.arange(n_tokens, device=token_ids.device)
        x = self.tok_emb(token_ids) + self.pos_emb(positions)  # [batch, n_tokens, 768]
        x = self.drop(x)
        x = self.blocks(x)                                     # 12 blocks, shape unchanged
        x = self.final_norm(x)
        return self.out_head(x)                                # [batch, n_tokens, 50257]

def generate_greedy(model, token_ids, max_new_tokens, context_size):
    for _ in range(max_new_tokens):
        window = token_ids[:, -context_size:]      # 1. crop
        with torch.no_grad():
            logits = model(window)                 # 2. [batch, n_tokens, vocab]
        last = logits[:, -1, :]                    # 3. next-token scores
        probs = torch.softmax(last, dim=-1)        # 4. probabilities...
        next_id = torch.argmax(probs, dim=-1, keepdim=True)  # ...and the winner
        token_ids = torch.cat([token_ids, next_id], dim=1)   # 5. append
    return token_ids

# ---- from Chapter 5 ----
import tiktoken
import torch

tokenizer = tiktoken.get_encoding("gpt2")

def text_to_token_ids(text, tokenizer):
    ids = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
    return torch.tensor(ids).unsqueeze(0)     # add a batch dimension

def token_ids_to_text(token_ids, tokenizer):
    ids = token_ids.squeeze(0).tolist()       # drop the batch dimension
    return tokenizer.decode(ids)

def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch = input_batch.to(device)
    target_batch = target_batch.to(device)
    logits = model(input_batch)
    return F.cross_entropy(logits.flatten(0, 1), target_batch.flatten())


def calc_loss_loader(data_loader, model, device, num_batches=None):
    if len(data_loader) == 0:
        return float("nan")
    n = len(data_loader) if num_batches is None else min(num_batches, len(data_loader))
    total = 0.0
    for i, (x, y) in enumerate(data_loader):
        if i >= n:
            break
        total += calc_loss_batch(x, y, model, device).item()
    return total / n

def train_model_simple(model, train_loader, val_loader, optimizer, device,
                       num_epochs, eval_freq, eval_iter, start_context, tokenizer):
    train_losses, val_losses, tokens_seen = [], [], []
    seen, step = 0, -1

    for epoch in range(num_epochs):
        model.train()                              # enable dropout
        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()                  # clear old gradients
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward()                        # compute new gradients
            optimizer.step()                       # update the weights
            seen += input_batch.numel()
            step += 1

            if step % eval_freq == 0:
                tr, va = evaluate_model(model, train_loader, val_loader, device, eval_iter)
                train_losses.append(tr)
                val_losses.append(va)
                tokens_seen.append(seen)
                print(f"epoch {epoch+1} step {step:05d}: train {tr:.3f}, val {va:.3f}")

        # after every epoch, look at what the model writes
        generate_and_print_sample(model, tokenizer, device, start_context)

    return train_losses, val_losses, tokens_seen

def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
        val_loss = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)
    model.train()
    return train_loss, val_loss


def generate_and_print_sample(model, tokenizer, device, start_context):
    model.eval()
    context_size = model.pos_emb.weight.shape[0]
    idx = text_to_token_ids(start_context, tokenizer).to(device)
    with torch.no_grad():
        out = generate_greedy(model, idx, max_new_tokens=40, context_size=context_size)
    print(token_ids_to_text(out, tokenizer).replace("\n", " "))
    model.train()

def generate(model, idx, max_new_tokens, context_size,
             temperature=0.0, top_k=None, eos_id=None):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]          # crop to the context window
        with torch.no_grad():
            logits = model(idx_cond)[:, -1, :]      # logits for the last position

        if top_k is not None:
            top_logits, _ = torch.topk(logits, top_k)
            cutoff = top_logits[:, -1:]
            logits = torch.where(logits < cutoff,
                                 torch.full_like(logits, float("-inf")), logits)

        if temperature > 0.0:
            probas = torch.softmax(logits / temperature, dim=-1)
            idx_next = torch.multinomial(probas, num_samples=1)
        else:
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)

        if eos_id is not None and (idx_next == eos_id).all():
            break                                   # stop at end-of-text
        idx = torch.cat((idx, idx_next), dim=1)
    return idx

import json
import os
import urllib.request
from safetensors.torch import load_file

GPT2_REPOS = {"124M": "gpt2", "355M": "gpt2-medium", "774M": "gpt2-large", "1558M": "gpt2-xl"}

def download_and_load_gpt2(model_size, models_dir="gpt2"):
    """Downloads GPT-2 weights once, then returns (settings, params) from disk."""
    folder = os.path.join(models_dir, model_size)
    os.makedirs(folder, exist_ok=True)
    for file in ("config.json", "model.safetensors"):
        path = os.path.join(folder, file)
        if not os.path.exists(path):
            url = f"https://huggingface.co/openai-community/{GPT2_REPOS[model_size]}/resolve/main/{file}"
            print("Downloading", url)
            urllib.request.urlretrieve(url, path)
    with open(os.path.join(folder, "config.json")) as f:
        settings = json.load(f)
    params = load_file(os.path.join(folder, "model.safetensors"))  # name -> tensor
    return settings, params

def assign(param, tensor):
    if param.shape != tensor.shape:
        raise ValueError(f"Shape mismatch: model {tuple(param.shape)} vs checkpoint {tuple(tensor.shape)}")
    return torch.nn.Parameter(tensor.detach().clone())

def load_weights_into_gpt(gpt, params):
    p = {k.removeprefix("transformer."): v for k, v in params.items()}
    gpt.tok_emb.weight = assign(gpt.tok_emb.weight, p["wte.weight"])
    gpt.pos_emb.weight = assign(gpt.pos_emb.weight, p["wpe.weight"])

    for i, block in enumerate(gpt.blocks):
        h = f"h.{i}."
        # OpenAI stores linear weights as (in, out); nn.Linear expects (out, in), hence .T
        # Queries, keys and values live in one combined matrix: split it in three.
        q_w, k_w, v_w = p[h + "attn.c_attn.weight"].chunk(3, dim=-1)
        q_b, k_b, v_b = p[h + "attn.c_attn.bias"].chunk(3, dim=-1)
        attn = block.attn
        attn.to_q.weight, attn.to_q.bias = assign(attn.to_q.weight, q_w.T), assign(attn.to_q.bias, q_b)
        attn.to_k.weight, attn.to_k.bias = assign(attn.to_k.weight, k_w.T), assign(attn.to_k.bias, k_b)
        attn.to_v.weight, attn.to_v.bias = assign(attn.to_v.weight, v_w.T), assign(attn.to_v.bias, v_b)
        attn.out_proj.weight = assign(attn.out_proj.weight, p[h + "attn.c_proj.weight"].T)
        attn.out_proj.bias = assign(attn.out_proj.bias, p[h + "attn.c_proj.bias"])

        expand, project = block.ff.net[0], block.ff.net[2]
        expand.weight = assign(expand.weight, p[h + "mlp.c_fc.weight"].T)
        expand.bias = assign(expand.bias, p[h + "mlp.c_fc.bias"])
        project.weight = assign(project.weight, p[h + "mlp.c_proj.weight"].T)
        project.bias = assign(project.bias, p[h + "mlp.c_proj.bias"])

        block.norm1.scale = assign(block.norm1.scale, p[h + "ln_1.weight"])
        block.norm1.shift = assign(block.norm1.shift, p[h + "ln_1.bias"])
        block.norm2.scale = assign(block.norm2.scale, p[h + "ln_2.weight"])
        block.norm2.shift = assign(block.norm2.shift, p[h + "ln_2.bias"])

    gpt.final_norm.scale = assign(gpt.final_norm.scale, p["ln_f.weight"])
    gpt.final_norm.shift = assign(gpt.final_norm.shift, p["ln_f.bias"])
    gpt.out_head.weight = assign(gpt.out_head.weight, p["wte.weight"])  # GPT-2 ties these

## Two kinds of fine-tuning

## Preparing the dataset

In [ ]:
import os
import urllib.request
import zipfile

url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
data_path = "sms_spam_collection/SMSSpamCollection.tsv"

if not os.path.exists(data_path):
    urllib.request.urlretrieve(url, "sms_spam_collection.zip")
    with zipfile.ZipFile("sms_spam_collection.zip") as archive:
        archive.extractall("sms_spam_collection")
    os.rename("sms_spam_collection/SMSSpamCollection", data_path)

In [ ]:
import pandas as pd

df = pd.read_csv(
    data_path,
    sep="\t", header=None, names=["Label", "Text"],
)
print(df["Label"].value_counts())
# ham     4825
# spam     747

### Balancing the classes

In [ ]:
def balance(df, seed=123):
    spam = df[df["Label"] == "spam"]
    ham = df[df["Label"] == "ham"].sample(len(spam), random_state=seed)
    return pd.concat([ham, spam])

balanced = balance(df)
balanced["Label"] = balanced["Label"].map({"ham": 0, "spam": 1})

### Train, validation and test splits

In [ ]:
def split(df, train_frac=0.7, val_frac=0.1, seed=123):
    df = df.sample(frac=1, random_state=seed).reset_index(drop=True)
    n_train = int(len(df) * train_frac)
    n_val = int(len(df) * val_frac)
    return (
        df[:n_train],
        df[n_train:n_train + n_val],
        df[n_train + n_val:],
    )

train_df, val_df, test_df = split(balanced)
for name, part in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    part.to_csv(f"{name}.csv", index=False)

### Padding to a common length

In [ ]:
import tiktoken
import torch
from torch.utils.data import Dataset

tokenizer = tiktoken.get_encoding("gpt2")
PAD_ID = 50256  # <|endoftext|>

class SpamDataset(Dataset):
    def __init__(self, csv_path, tokenizer, max_length=None):
        self.df = pd.read_csv(csv_path)
        encoded = [tokenizer.encode(t) for t in self.df["Text"]]

        # Use the longest message unless a length is imposed (for val/test)
        self.max_length = max_length or max(len(ids) for ids in encoded)
        self.inputs = []
        for ids in encoded:
            ids = ids[:self.max_length]                      # truncate if too long
            ids += [PAD_ID] * (self.max_length - len(ids))   # pad if too short
            self.inputs.append(ids)

    def __getitem__(self, idx):
        x = torch.tensor(self.inputs[idx])
        y = torch.tensor(self.df.iloc[idx]["Label"])
        return x, y

    def __len__(self):
        return len(self.df)

train_ds = SpamDataset("train.csv", tokenizer)
val_ds = SpamDataset("validation.csv", tokenizer, max_length=train_ds.max_length)
test_ds = SpamDataset("test.csv", tokenizer, max_length=train_ds.max_length)
print(train_ds.max_length)  # 120

In [ ]:
from torch.utils.data import DataLoader

torch.manual_seed(123)
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=8)
test_loader = DataLoader(test_ds, batch_size=8)

x, y = next(iter(train_loader))
print(x.shape, y.shape)  # torch.Size([8, 120]) torch.Size([8])

## Starting from pretrained weights

In [ ]:
BASE_CONFIG = {
    "vocab_size": 50257,
    "context_length": 1024,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.0,
    "qkv_bias": True,   # the OpenAI checkpoint uses biases in Q, K, V
}

settings, params = download_and_load_gpt2(model_size="124M", models_dir="gpt2")
model = GPTModel(BASE_CONFIG)
load_weights_into_gpt(model, params)
model.eval()

In [ ]:
ids = generate_greedy(
    model, text_to_token_ids("The weather today is", tokenizer),
    max_new_tokens=15, context_size=BASE_CONFIG["context_length"],
)
print(token_ids_to_text(ids, tokenizer))
# The weather today is ... (followed by fluent, if aimless, text)

In [ ]:
prompt = (
    "Answer 'spam' or 'not spam'. Message: "
    "'Congratulations! You have won a free cruise. Reply YES to claim.' Answer:"
)

## Adding a classification head

### Freezing most of the network

In [ ]:
# 1. Freeze every existing parameter
for p in model.parameters():
    p.requires_grad = False

# 2. Swap in a new head. Fresh layers are trainable by default.
torch.manual_seed(123)
num_classes = 2
model.out_head = torch.nn.Linear(BASE_CONFIG["emb_dim"], num_classes)

# 3. Unfreeze the last transformer block and the final LayerNorm
for p in model.blocks[-1].parameters():
    p.requires_grad = True
for p in model.final_norm.parameters():
    p.requires_grad = True

### Which output do we use?

In [ ]:
inputs = torch.tensor(tokenizer.encode("Are you free tonight")).unsqueeze(0)
with torch.no_grad():
    outputs = model(inputs)
print(outputs.shape)  # torch.Size([1, 4, 2])

In [ ]:
last_logits = outputs[:, -1, :]
print(last_logits)  # tensor([[ 0.8121, -1.3406]])  (random head, for now)

## Loss and accuracy

In [ ]:
probs = torch.softmax(last_logits, dim=-1)
label = torch.argmax(probs, dim=-1)   # same as argmax(last_logits)

### Accuracy

In [ ]:
def accuracy(loader, model, device, num_batches=None):
    model.eval()
    correct, seen = 0, 0
    num_batches = min(num_batches or len(loader), len(loader))
    for i, (x, y) in enumerate(loader):
        if i >= num_batches:
            break
        x, y = x.to(device), y.to(device)
        with torch.no_grad():
            preds = model(x)[:, -1, :].argmax(dim=-1)
        correct += (preds == y).sum().item()
        seen += y.shape[0]
    return correct / seen

### Loss

In [ ]:
def loss_batch(x, y, model, device):
    x, y = x.to(device), y.to(device)
    logits = model(x)[:, -1, :]            # [batch, 2]
    return torch.nn.functional.cross_entropy(logits, y)

def loss_loader(loader, model, device, num_batches=None):
    num_batches = min(num_batches or len(loader), len(loader))
    total = 0.0
    for i, (x, y) in enumerate(loader):
        if i >= num_batches:
            break
        total += loss_batch(x, y, model, device).item()
    return total / num_batches

## Fine-tuning and evaluating

In [ ]:
def train_classifier(model, train_loader, val_loader, optimizer, device,
                     num_epochs, eval_freq=50, eval_iter=5):
    step = 0
    for epoch in range(num_epochs):
        model.train()
        for x, y in train_loader:
            optimizer.zero_grad()
            loss = loss_batch(x, y, model, device)
            loss.backward()
            optimizer.step()
            step += 1

            if step % eval_freq == 0:
                model.eval()
                with torch.no_grad():
                    tr = loss_loader(train_loader, model, device, eval_iter)
                    va = loss_loader(val_loader, model, device, eval_iter)
                model.train()
                print(f"epoch {epoch+1} step {step}: train {tr:.3f}, val {va:.3f}")

        tr_acc = accuracy(train_loader, model, device, eval_iter)
        va_acc = accuracy(val_loader, model, device, eval_iter)
        print(f"  accuracy: train {tr_acc:.0%} | val {va_acc:.0%}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.1)
train_classifier(model, train_loader, val_loader, optimizer, device, num_epochs=5)

### The final score

In [ ]:
for name, loader in [("train", train_loader), ("val", val_loader), ("test", test_loader)]:
    print(f"{name}: {accuracy(loader, model, device):.1%}")
# Typical results (yours will differ slightly):
# train: 97.2%
# val: 97.3%
# test: 95.7%

### Classifying new messages

In [ ]:
def classify(text, model, tokenizer, device, max_length=120):
    model.eval()
    ids = tokenizer.encode(text)[:max_length]
    ids += [PAD_ID] * (max_length - len(ids))
    x = torch.tensor(ids, device=device).unsqueeze(0)
    with torch.no_grad():
        label = model(x)[:, -1, :].argmax(dim=-1).item()
    return "spam" if label == 1 else "not spam"

print(classify("Congratulations! You have won a free cruise. "
               "Reply YES now to claim your prize.", model, tokenizer, device))
# spam
print(classify("Running ten minutes late, can you save me a seat?",
               model, tokenizer, device))
# not spam

In [ ]:
torch.save(model.state_dict(), "spam_classifier.pth")